# 03 · News — vector RAG (semantic, filtered, hybrid, cited)

`AgensgraphVectorStore` as a LlamaIndex `VectorStoreIndex` over news articles.
This notebook tours four retrieval modes, the richer metadata-filter operators,
and the full store-mutation lifecycle.

> Run `ingest.py` first to build the `news` graph.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

configure_settings()
from llama_index.core import VectorStoreIndex
store = agens.make_vector_store(graph_name="news", node_label="Article")
index = VectorStoreIndex.from_vector_store(store, embed_model=get_embed_model())
question = "What are companies doing with artificial intelligence?"

## (a) Plain semantic search

HNSW nearest-neighbour over the chunk embeddings.

In [2]:
def show(hits):
    for h in hits:
        m = h.node.metadata or {}
        print(f"{h.score:.3f}  {m.get('domain','?')} · {m.get('date','?')} · {(m.get('title') or '')[:55]}")
show(index.as_retriever(similarity_top_k=5).retrieve(question))

0.650  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.637  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.621  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.620  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.619  www.techrepublic.com · 2017-12-11 · Why some of the world's biggest companies are using AI 


## (b) Metadata-filtered retrieval

`MetadataFilters` translate to an indexed Cypher `WHERE` — here `domain IN [...]`
`AND date >= 2017-01-01`.

In [3]:
from llama_index.core.vector_stores import (MetadataFilters, MetadataFilter,
                                             FilterOperator, FilterCondition)
domains = [r["domain"] for r in store.database_query(
    'MATCH (n:"Article") WHERE n.domain IS NOT NULL '
    'RETURN n.domain AS domain, count(*) AS c ORDER BY c DESC LIMIT 4')]
filters = MetadataFilters(condition=FilterCondition.AND, filters=[
    MetadataFilter(key="domain", operator=FilterOperator.IN, value=domains),
    MetadataFilter(key="date", operator=FilterOperator.GTE, value="2017-01-01")])
print("domains:", domains)
show(index.as_retriever(similarity_top_k=5, filters=filters).retrieve(question))

domains: ['nationalpost.com', 'www.taiwannews.com.tw', 'www.nigeriatoday.ng', 'abcnews.go.com']


0.541  www.taiwannews.com.tw · 2018-04-24 · Splunk Customers Accelerate Business Value Through Arti
0.538  www.taiwannews.com.tw · 2018-04-25 · Global Artificial Intelligence Market in Education Sect
0.534  www.taiwannews.com.tw · 2018-04-25 · Global Artificial Intelligence Market in Education Sect
0.522  www.taiwannews.com.tw · 2018-04-24 · Splunk Customers Accelerate Business Value Through Arti


## (b2) Richer operators — NIN / CONTAINS / OR / nested groups

`MetadataFilters` support the full operator set and nested boolean groups, all
translated to an injection-safe Cypher `WHERE`.

In [4]:
# exclude the top hit's domain (NIN), require a url substring (CONTAINS), and a
# nested OR — a result set visibly different from (b).
top_domain = (index.as_retriever(similarity_top_k=1).retrieve(question)[0].node.metadata or {}).get("domain")
rich = MetadataFilters(condition=FilterCondition.AND, filters=[
    MetadataFilter(key="domain", operator=FilterOperator.NIN, value=[top_domain]),
    MetadataFilters(condition=FilterCondition.OR, filters=[
        MetadataFilter(key="url",   operator=FilterOperator.CONTAINS, value="http"),
        MetadataFilter(key="title", operator=FilterOperator.TEXT_MATCH, value="AI")])])
print("excluding top domain:", top_domain,
      "\nsupported: EQ NE GT GTE LT LTE IN NIN CONTAINS TEXT_MATCH ANY ALL IS_EMPTY")
show(index.as_retriever(similarity_top_k=5, filters=rich).retrieve(question))

excluding top domain: www.techrepublic.com 
supported: EQ NE GT GTE LT LTE IN NIN CONTAINS TEXT_MATCH ANY ALL IS_EMPTY


0.578  www.sci-tech-today.com · 2018-01-28 · Tech Firms Race To Create Smarter Thinking Computers
0.565  www.sci-tech-today.com · 2018-01-28 · Tech Firms Race To Create Smarter Thinking Computers
0.550  www.industryweek.com · 2017-12-11 · Teaching Self-Learning Machines to Forget
0.547  www.yahoo.com · 2017-12-11 · A Team of MIT Scientists Taught an AI to Get Emotional 
0.547  www.digitaltrends.com · 2017-02-15 · Can Augmentation Save Workers from Job Automation?


## (c) Hybrid search (vector + keyword RRF)

A `hybrid_search=True` store fuses HNSW semantic search with full-text keyword
search by reciprocal rank fusion (each modality uses its own index).

In [5]:
hstore = agens.make_vector_store(graph_name="news", node_label="Article", hybrid_search=True)
hindex = VectorStoreIndex.from_vector_store(hstore, embed_model=get_embed_model())
show(hindex.as_retriever(similarity_top_k=5, vector_store_query_mode="hybrid").retrieve(question))

0.016  www.techrepublic.com · 2018-02-02 · Top 5: Things AI might actually be good for
0.016  nationalpost.com · 2018-04-24 · Aeroplan's troublesome 'purity of the country' survey i
0.016  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI
0.016  uproxx.com · 2018-04-24 · China Plans To Implement Social Credit To 1.4 Billion C
0.016  www.techrepublic.com · 2018-02-02 · 3 ways to reshape your workforce in the age of AI


## (d) Cited RAG

`CitationQueryEngine` answers with inline `[N]` markers tied to sources.

In [6]:
from llama_index.core.query_engine import CitationQueryEngine
resp = CitationQueryEngine.from_args(index, similarity_top_k=5).query(question)
print(str(resp).strip())
print("\nSources:")
for i, s in enumerate(resp.source_nodes, 1):
    m = s.node.metadata or {}
    print(f"  [{i}] {m.get('domain','?')} · {(m.get('title') or '')[:55]}")

Companies are leveraging artificial intelligence (AI) in various ways to enhance their operations and improve efficiency. For instance, AI is being used to automate the scanning of websites for suspicious content to help stop predators, as seen in initiatives by the National Center for Missing and Exploited Children [1]. Additionally, AI is playing a significant role in the recruitment process, where it helps sort through resumes and analyze candidates' responses, thereby reducing the time to hire and increasing acceptance rates, as demonstrated by Unilever's use of the AI tool HireVue [1]. 

Moreover, AI is also being utilized in customer service to assist human agents by processing natural language and routing inquiries effectively [1]. In the agricultural sector, AI tools like FarmLogs analyze complex data to optimize crop yields [4]. Overall, businesses are optimistic about the potential of AI to create net job gains and improve market differentiation, with many executives believin

## (e) Store lifecycle — add / get_nodes / delete_nodes / delete / clear

A scratch `news_crud_demo` graph with constant placeholder embeddings (no OpenAI
calls): add nodes, read them back by filter, delete by filter and by source
document, then clear — the populated `news` graph is untouched.

In [7]:
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo
crud = agens.make_vector_store(graph_name="news_crud_demo", node_label="Doc")  # scratch graph
crud.clear()
def mk(nid, doc, dom):
    n = TextNode(id_=nid, text=f"scratch {nid}", embedding=[0.1]*EMBED_DIM, metadata={"domain": dom})
    n.relationships[NodeRelationship.SOURCE] = RelatedNodeInfo(node_id=doc); return n
crud.add([mk("c1", "docA", "a.com"), mk("c2", "docA", "a.com"), mk("c3", "docB", "b.com")])
cnt = lambda: crud.database_query('MATCH (n:"Doc") RETURN count(*) AS c')[0]["c"]
a_com = MetadataFilters(filters=[MetadataFilter(key="domain", operator=FilterOperator.EQ, value="a.com")])
print("added:", cnt(), "| get_nodes(domain=a.com):", [n.node_id for n in crud.get_nodes(filters=a_com)])
crud.delete_nodes(filters=MetadataFilters(
    filters=[MetadataFilter(key="domain", operator=FilterOperator.EQ, value="b.com")]))
print("after delete_nodes(domain=b.com):", cnt())
crud.delete(ref_doc_id="docA"); print("after delete(ref_doc_id=docA):", cnt())
crud.clear(); print("after clear():", cnt())

added: 3 | get_nodes(domain=a.com): ['c1', 'c2']
after delete_nodes(domain=b.com): 2
after delete(ref_doc_id=docA): 0
after clear(): 0


## How it was built

`ingest.py` chunks CC-News, embeds in parallel, and `async_add`s with metadata;
it indexes the filterable keys so filtered search stays fast:

```python
nodes = SentenceSplitter(chunk_size=256).get_nodes_from_documents(docs)
await store.async_add(nodes)            # nodes carry {domain, date, title, url}
store.create_property_index("domain"); store.create_property_index("date")
```

In [8]:
agens.close()